**Lab2 Assignment**


*Agustín* *Prieto*

*Jesús Sanz*


In [28]:
import heapq
import numpy as np
import sys
from abc import ABC,abstractmethod
from enum import Enum, auto
import json
import os
import time
import random

Class **Solution**

In [29]:
class Solution:
    momento = 0
    def __init__(self,score,solution):
        self.score = score
        self.solution = solution
        self.momento = Solution.momento
        Solution.momento += 1
    def __lt__(self,obj):
        if self.score == obj.score:
            return self.momento < obj.momento
        return self.score < obj.score

Class **Node**

In [30]:
class Node:
    def __init__(self,parent,state,action,depth,accumulatedCost):
        self.parent = parent
        self.state = state # we want it to be type State
        self.action = action # the same with action is going to be a class
        self.depth = depth
        self.accumulatedCost = accumulatedCost # this way it's easier to compute g(n) on f(n) = g(n)+h(n)
        self.momento = 0
    def __str__(self):
        stringToReturn = (
        f'parent --> { self.parent}\n'
        f'state --> { self.state.state}\n'
        f'action --> (origin, destination,cost) --> ({self.action.origin} , {self.action.destination}, {self.action.cost})\n'
        f'depth -->  {self.depth}'
        )
        return stringToReturn
    def __lt__(self,obj):
        return self.momento < obj.momento

Class **State**

In [31]:
class State:
    def __init__(self,state,longitude,latitude):
        self.state = state
        self.longitude = longitude
        self.latitude = latitude

Class **Selection**

In [32]:
class Selection(Enum):
    REPLACEMENT = auto()
    TRUNCATION = auto()
    ELITISM = auto()

Class **Action**

In [33]:
class Action: 
    def __init__(self,origin,destination,cost):
        self.origin = origin
        self.destination = destination
        self.cost = cost
    def __str__(self):
       return(
           f' {self.origin} → {self.destination} ({self.cost})'
       )

Class **Problem**

In [34]:
class Problem:
    def __init__(self,file_name):
        with open(file_name,'r') as file:
            self.dictionary = json.load(file)
        self.dictionary['intersections'] = {inter['identifier']: inter for inter in self.dictionary.get('intersections')}# O(m)
        self.dictionary['candidates'] = {candidate[0]:{"identifier":candidate[0],"population":candidate[1]} for candidate in self.dictionary.get('candidates')}
        self.dictionary['maxSpeedOfAllSpeeds'] = float('-inf') 
        # Add the 'whereto' attribute to each intersection
        for inter in self.dictionary['intersections'].values():# O(m)
            inter['whereto'] = []

        # Populate the 'whereto' attribute based on the segments
        for segment in self.dictionary.get('segments'):# O(n)
            origin = segment['origin']#O(1)
            destination = segment['destination'] #O(1)
            distance = segment['distance'] #O(1)
            speed_kmh = segment['speed'] # O(1)

            # Convert speed from km/h to m/s
            speed_ms = speed_kmh * (1000 / 3600)
            #self.dictionary['mostRepeatedSpeed'].append(speed_ms)
            
            if(speed_ms > self.dictionary.get('maxSpeedOfAllSpeeds')):#O(1)
                self.dictionary['maxSpeedOfAllSpeeds'] = speed_ms
    
            # Calculate the cost
            cost = distance / speed_ms
    
            # Add the destination and cost to the 'whereto' attribute of the origin intersection
            #if origin in self.dictionary.get('intersections'): # O(1)
            self.dictionary.get('intersections').get(origin).get('whereto').append({'id': destination, 'cost': cost})
        for i in self.dictionary.get('intersections'):
            self.dictionary['intersections'][i]['whereto'] = sorted(self.dictionary.get('intersections').get(i).get('whereto'),key = lambda x:x['id'])

Class **Search**

In [35]:
class Search(ABC):
    initial = 0
    final = 0
    a_star_total = 0
    a_star_real = 0
    evaluated_real = 0
    evaluated_total = 0
    time = dict()
    individuals_already_evaluated = dict()
    def __init__(self,problem):
        Search.evaluated_total = 0
        Search.evaluated_real = 0
        self.problem = problem
        self.openDS = [] # open_data_structure
        self.explored = {0}
        self.nodesGenerated = 0
    def insert(self,element):  
        self.openDS.append(element)
    def extract():
        pass
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       evaluation
    #////////////////////////////////////////////////////////////////////
    def evaluation(self,solution):#O(n^2)
        if tuple(solution) in Search.individuals_already_evaluated:
            Search.evaluated_total += 1
            return Search.individuals_already_evaluated.get(tuple(solution))
        total_population = 0 
        weight_per_candidate = 0
        
        # 1. Iterate through each candidate
        for idx,i in enumerate(self.problem.dictionary.get('candidates').values()): # O(n^2)
            min_of_all_a_star = 3600*5
            pop = i.get('population')
            total_population += pop 
            candidate = i.get('identifier') 
            Search.initial = candidate
            if Main.DEBUG_EVALUATION:
                print(f'place_id={candidate};citizens={pop}')
            # 2. Calculate the time from each candidate to a fixed station
            for idj,j in enumerate(self.problem.dictionary.get('candidates').values()): # O(n) 
                if solution[idj] == 0:
                    continue 
                station = j.get('identifier')
                Search.final = station 
                time_a_star = self.get_time_a_star() 
                if Main.DEBUG_EVALUATION:
                    print(f'to station with id {station} = {time_a_star}')
                if time_a_star < min_of_all_a_star:
                    min_of_all_a_star = time_a_star
            if min_of_all_a_star == 3600*5:
                continue
            if Main.DEBUG_EVALUATION:
                print(f'min_distance --> {min_of_all_a_star}')
            only_this_one = min_of_all_a_star * pop
            weight_per_candidate += only_this_one
            if Main.DEBUG_EVALUATION:
                print(f'accounting for ={only_this_one}')
                print(f'__________________________________________')
        Search.evaluated_real += 1
        Search.evaluated_total += 1
        Search.individuals_already_evaluated[tuple(solution)] = weight_per_candidate / total_population
        return Search.individuals_already_evaluated[tuple(solution)]
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       get_time_a_star
    #////////////////////////////////////////////////////////////////////
    def get_time_a_star(self):
        Search.a_star_total += 1
        if( not self.is_already_in_memory()): # if it is NOT in memory
            # save it in memory
            return self.save_changes()
        #otherwise just get it from memory
        return Search.time.get((Search.initial,Search.final))
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       is_already_in_memory
    #////////////////////////////////////////////////////////////////////
    def is_already_in_memory(self):
        return (Search.initial,Search.final) in Search.time
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       save_changes
    #////////////////////////////////////////////////////////////////////
    def save_changes(self):
        Search.a_star_real += 1
        instance_of_search = AStar(self.problem)
        time_a_star = instance_of_search.search()
        Search.time[(Search.initial,Search.final)] = time_a_star
        return time_a_star
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       search
    #////////////////////////////////////////////////////////////////////
    @abstractmethod
    def search(self):
       pass
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       generateARandomSolution
    #////////////////////////////////////////////////////////////////////
    def generateARandomSolution(self):
        length_of_array_of_candidates = len(self.problem.dictionary.get('candidates'))
        number_of_ones_we_need = self.problem.dictionary.get('number_stations')
        random_solution = np.zeros(length_of_array_of_candidates,dtype=int)
        positions_where_we_are_going_to_introduce_ones = np.random.choice(length_of_array_of_candidates,number_of_ones_we_need,replace = False)
        random_solution[positions_where_we_are_going_to_introduce_ones] = 1
        return random_solution
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       expand
    #////////////////////////////////////////////////////////////////////
    def expand(self,Node_param): #O(n)
        successors = []
        currentIntersection = self.problem.dictionary.get('intersections').get(Node_param.state.state) # O(1)

        for destination in currentIntersection.get('whereto'): # O(n)
            """currentIntersection.get("whereto")
            [{'id': 1256026663, 'cost': 1.7331}, {'id': 1531659796, 'cost': 2.346}]"""
            if destination.get('id') in self.explored: # O(1) 
                continue
            newAction = Action(# O(1)
                    Node_param.state.state, 
                    destination.get("id"), 
                    destination.get("cost") 
                )
            # REMEMBER THAT destination is A DICTIONARY {"id":,"cost":}
            newState = State(newAction.destination,self.problem.dictionary.get('intersections').get(destination.get('id')).get('longitude'),self.problem.dictionary.get('intersections').get(destination.get('id')).get('latitude'))
            newNode = Node(Node_param,newState,newAction,Node_param.depth+1,Node_param.accumulatedCost+newAction.cost)
            self.nodesGenerated+=1
            newNode.momento = self.nodesGenerated
            successors.append(newNode)
        return successors
    

Class **AStar**

In [36]:
class AStar(Search):# takes into account g(n), not only h(n)
    ##############################################################################################
    ############################################# insert #########################################
    ##############################################################################################
    def __init__(self,problem):
        self.distance = dict()
        super().__init__(problem)
    def insert(self,element):
        goalId = Search.final
        self.openDS = list(self.openDS)
        heuristic = (self.computeHeuristic(element,goalId)/self.problem.dictionary.get('maxSpeedOfAllSpeeds'))+element.accumulatedCost
        heapq.heappush(self.openDS,(heuristic,element))
    ##############################################################################################
    ############################################ extract #########################################
    ##############################################################################################
    def extract(self): #O(1)
        return heapq.heappop(self.openDS)[1] 
    ##############################################################################################
    ######################################## computeHeuristic ####################################
    ##############################################################################################
    def computeHeuristic(self,node_param,goalId): #O(1)
        self.openDS = list(self.openDS) 
        coord_1 = (
            node_param.state.longitude,
              node_param.state.latitude
              ) 
        coord_2 = (
            self.problem.dictionary.get('intersections').get(goalId).get('longitude'), 
        self.problem.dictionary.get('intersections').get(goalId).get('latitude')
        )   
        if (coord_1,coord_2) in self.distance:
            return self.distance.get((coord_1,coord_2))
        resultado = self.haversine(coord_1,coord_2)
        self.distance[(coord_1,coord_2)] = resultado 
        distancia = resultado * 1000
        return distancia

#////////////////////////////////////////////////////////////////////
#                   haversine
#\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    
    def haversine(self,coord1,coord2):
        lon1,lat1=coord1
        lon2,lat2=coord2
        import math
        R=6371000                               # radius of Earth in meters
        phi_1=math.radians(lat1)
        phi_2=math.radians(lat2)

        delta_phi=math.radians(lat2-lat1)
        delta_lambda=math.radians(lon2-lon1)

        a=math.sin(delta_phi/2.0)**2+\
           math.cos(phi_1)*math.cos(phi_2)*\
           math.sin(delta_lambda/2.0)**2
        c=2*math.atan2(math.sqrt(a),math.sqrt(1-a))
        
        meters=R*c                         # output distance in meters
        km=meters/1000.0              # output distance in kilometers
        return km
    ##############################################################################################
    ######################################### search #########################################
    ##############################################################################################
    def search(self): # O(n) 
        self.initializeOpen(Search.initial) # O(1)
        self.insert(self.root)
        while len(self.openDS)!=0:    
            node = self.extract() # O(1) 
            if node.state.state not in self.explored:
                if(self.testGoal(node)): 
                    self.depth = node.depth
                    self.totalCost = node.accumulatedCost
                    return node.accumulatedCost 
                successors1 = self.expand(node) # O(n)
                for  successor in successors1: # O(n)
                    self.insert(successor) # O(1)
                self.explored.add(node.state.state) 
        return 3600*5
    ##############################################################################################
    ######################################## initializeOpen ######################################
    ##############################################################################################
    def initializeOpen(self,initial): # O(1)
        longitudeInitialNode = self.problem.dictionary.get('intersections').get(initial).get('longitude')
        latitudeInitialNode = self.problem.dictionary.get('intersections').get(initial).get('latitude')
        self.root = Node(None,State(initial,longitudeInitialNode,latitudeInitialNode),Action(None,initial,0),0,0) 
        self.nodesGenerated+=1
        self.root.momento = self.nodesGenerated
    #################################################################################
    ####################             testGoal              ##########################
    #################################################################################
    def testGoal(self,node):# O(1)
        return Search.final == node.state.state

Class **GeneticAlgorithm**

In [37]:
#\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
#                       GeneticAlgorithm
#////////////////////////////////////////////////////////////////
class GeneticAlgorithm(Search):
    def __init__(self,problem):
        self.p = []
        self.p_ = []
        Search.a_star_real = 0
        Search.a_star_total = 0
        super().__init__(problem)
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       search
    #/////////////////////////////////////////////////////////////////////
    
    def search(self,population_size = Main.POPULATION_SIZE):
        #Lesson 8 Slide 18
        
        self.p = self.generate_population(population_size) # O(n) # create candidate solutions (individuals)
        self.p = self.evaluate(self.p) # obtains  their score
        

        number_of_generations = Main.NUM_GENERATIONS
        while(number_of_generations != 0):# O(n)
            self.p_ = self.select_population(self.p) # Selects some individuals by score
            self.p_ = self.crossover(self.p_) #crosses pairs of selected individuals
            self.p_ = self.mutation(self.p_) # mutates the crossed individuals
            self.p_ = self.evaluate(self.p_) # obtains the score of the new individuals
            self.p = self.combine(self.p,self.p_) # forms the new generation            
            number_of_generations -= 1
        return self.p
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       generate_population
    #////////////////////////////////////////////////////////////////////
    
    def generate_population(self,population_size):#O(n)
        population = []
        for _ in range(population_size):#O(n)
            heapq.heappush(population,Solution(0,self.generateARandomSolution()))
        return population
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                           evaluate
    #/////////////////////////////////////////////////////////////////////
    def evaluate(self,population):
        total = 0
        list_for_heapq = []
        evaluation_values_not_to_be_calculated_again = []
        for i in range(len(population)):# O(n)
            evaluation_values_not_to_be_calculated_again.append(self.evaluation(population[i].solution))
            total += evaluation_values_not_to_be_calculated_again[i]
        for i in range(len(population)):# O(n)
            evaluation_value = evaluation_values_not_to_be_calculated_again[i]
            heapq.heappush(list_for_heapq,Solution(evaluation_value,population[i].solution))
        return list_for_heapq
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                               select_population
    #/////////////////////////////////////////////////////////////////////////
    def select_population(self,population):
        length_of_the_list_of_individuals = len(population)
        new_population = []
        # 1. Take k individuals randomly
        k = Main.K
        for _ in range(len(population)):
            # 2. Play the tournament
            tournament = []
            for _ in range(k):
                take_this_population = np.random.randint(0,length_of_the_list_of_individuals)
                heapq.heappush(tournament,population[take_this_population])
            new_population.append(heapq.heappop(tournament))
        return new_population
        
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                               crossover
    #//////////////////////////////////////////////////////////////////////
    def crossover(self,population):
        if len(population) % 2 != 0:
            population = population[:-1]
        first_half = []
        for i in range(int(len(population)/2)):
            first_half.append(population[0])
            population = population[1:]
        final_crossover = []
        for i in range(len(population)):
            list_of_two_children = self.join_these_two(population[i].solution,first_half[i].solution)
            population[i].solution = list_of_two_children[0]
            first_half[i].solution = list_of_two_children[1]
            final_crossover.append(population[i])
            final_crossover.append(first_half[i])
        return final_crossover
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                               join_these_two
    #//////////////////////////////////////////////////////////////////////
    def join_these_two(self,parent_one,parent_two):
        # 1. Pick a random split
        places_in_which_we_can_split = len(self.problem.dictionary.get('candidates'))-1
        point_in_which_we_split = np.random.choice(range(places_in_which_we_can_split))+1
        #2. Cross parents
        list_of_children = []
        first_child = np.concatenate((parent_one[:point_in_which_we_split] ,parent_two[point_in_which_we_split:]),axis=None)
        first_child = self.correctPossibleNumberStations(first_child)
        list_of_children.append(first_child)
        second_child =np.concatenate(( parent_one[point_in_which_we_split:] , parent_two [:point_in_which_we_split]),axis=None)
        second_child = self.correctPossibleNumberStations(second_child)
        list_of_children.append(second_child)
        return list_of_children
      ##############################################################################################
    ############################## correctPossibleNumberStations #################################
    ##############################################################################################
    def correctPossibleNumberStations(self,configuration):
        """
        In this function we correct the number of ceros and ones to our needs.
        :param configuration: ndarray from numpy library representing a binary array, a configuration, a solution, a chromosome
        """
        # we get the number of ones
        number_of_ones_we_must_have = self.problem.dictionary.get('number_stations')
        number_of_ones_we_have = np.count_nonzero(configuration==1)
        if (number_of_ones_we_have < number_of_ones_we_must_have):
            # we must introduce some ones
            difference = number_of_ones_we_must_have - number_of_ones_we_have
            positions_where_zeros_are = np.where(configuration == 0)[0]
            new_positions_ones = np.random.choice(positions_where_zeros_are,difference, replace = False)
            configuration[new_positions_ones] = 1
        if (number_of_ones_we_have > number_of_ones_we_must_have):
            # we must remove some ones
            difference =  number_of_ones_we_have - number_of_ones_we_must_have
            positions_where_ones_are = np.where(configuration==1)[0]
            new_positions_zero = np.random.choice(positions_where_ones_are,difference,replace = False)
            configuration[new_positions_zero] = 0
        return configuration
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       mutation
    #//////////////////////////////////////////////////////////////////////
    def mutation(self,population):
        mutation_rate = Main.MUTATION_RATE
        for i in range(len(population)): # going through solutions
            if random.uniform(0,1) <= mutation_rate:
                positions_where_zeros_are = np.where(population[i].solution == 0)[0]
                in_this_position_there_is_a_ZERO = np.random.choice(positions_where_zeros_are,1, replace = False)
                j = in_this_position_there_is_a_ZERO
                population[i].solution[j] = 1 # mutate gene
                positions_where_ones_are = np.where(population[i].solution == 1)[0]
                in_this_position_there_is_a_ONE = np.random.choice(positions_where_ones_are,1,replace = False)
                j = in_this_position_there_is_a_ONE
                population[i].solution[j] = 0  # mutate gene
        return population
    #\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    #                       combine
    #/////////////////////////////////////////////////////////////////////////
    def combine(self,population_one,population_two,strategy_combine = Selection.TRUNCATION):
        bag_of_individuals = []
        if  len(population_one) % 2 != 0:
            bag_of_individuals.append(heapq.heappop(population_one))
        match(strategy_combine):
            case Selection.REPLACEMENT:
                return population_two
            case Selection.ELITISM:
                for _ in range(int(len(population_two)-1)):
                    bag_of_individuals.append(heapq.heappop(population_two))
                bag_of_individuals.append(heapq.heappop(population_one))
                return bag_of_individuals
            case Selection.TRUNCATION:
                for _ in range(int(len(population_one)/2)):
                    bag_of_individuals.append(heapq.heappop(population_one))
                    bag_of_individuals.append(heapq.heappop(population_two))
                return bag_of_individuals
            case _: 
                return population_two

Class **RandomSearch**

In [38]:
class RandomSearch(Search):
    def __init__(self,problem):
        Search.a_star_real = 0
        Search.a_star_total = 0
        super().__init__(problem)
    def search(self,MaxIters = Main.NUM_GENERATIONS):
        iteration = 0
        list_of_solutions_with_corresponding_score = []
        while (iteration < MaxIters):
            x =Solution(0,self.generateARandomSolution())
            score = self.evaluation(x.solution)
            x.score = score
            heapq.heappush(list_of_solutions_with_corresponding_score,x)
            iteration += 1
        return list_of_solutions_with_corresponding_score

Class **Main**

In [39]:
class Main:
    DEBUG_EVALUATION = False
    DEBUG_MAIN = True
    POPULATION_SIZE = 500
    NUM_GENERATIONS = 100
    MUTATION_RATE = 0.2
    K = 3
    def main(self):
        for j in ['small','medium']:
            directory = 'C:\\googleMapsVS\\Google-Maps\\Lab2\\sample-problems-lab2\\'+j
            files = os.listdir(directory)
            if Main.DEBUG_MAIN:
                print(j)
            for i in files:
                os.chdir(directory)
                problem = Problem(i)
                for k in [RandomSearch(problem),GeneticAlgorithm(problem)]:
                    if Main.DEBUG_MAIN:
                        print(f'num_candidates is {len(problem.dictionary.get('candidates'))}, to select {problem.dictionary.get('number_stations')}')
                    start = time.perf_counter()
                    if Main.DEBUG_MAIN:
                        print(f'Problem {i}')
                        print("\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\")
                        print(f"\t \t {k.__class__.__name__}") 
                        print("//////////////////////////////////////////")
                    search = k
                    solution = search.search()
                    solution = heapq.heappop(solution)
                    end = time.perf_counter()
                    if Main.DEBUG_MAIN:
                        execution_time = self.format_seconds(end - start)
                        print(f'total seconds (Execution time) is : {execution_time}')
                        print(f'Best Solution : {solution.solution}')
                        print(f'fitness = {solution.score}')
                        print(f'The {problem.dictionary.get('number_stations')} stations will be located in intersections:')
                        print(solution.solution)
                        print()
                        print(f'The following are the stations ')
                        print('[',end=' ')
                        for id,i_ in enumerate(search.problem.dictionary.get('candidates').values()):
                            if  solution.solution[id] == 0:
                                continue
                            print(id,end=' ')
                        print(']')
                        print('\n')
                        print('A_Star calls:')
                        print(f'\ttotal --> {Search.a_star_total}')
                        print(f'\treal --> {Search.a_star_real}')
                        print('Evaluated individuals:')
                        print(f'\ttotal --> {Search.evaluated_total}')
                        print(f'\treal --> {Search.evaluated_real}')
                        print('___________________________________________')
                        print('\n')
    def format_seconds(self,seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        seconds_left = seconds % 60
        return f"{hours:02}:{minutes:02}:{seconds_left:02}"

In [ ]:
agus = Main()
agus.main()

small
num_candidates is 75, to select 7
Problem calle_agustina_aroca_albacete_250_0_candidates_75_ns_7.json
\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
	 	 RandomSearch
//////////////////////////////////////////
total seconds (Execution time) is : 00:00:4.433115200139582
Best Solution : [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0]
fitness = 14.181263097559093
The 7 stations will be located in intersections:
[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0]

The following are the stations 
[ 8 15 31 38 42 53 70 ]


A_Star calls:
	total --> 52500
	real --> 5625
Evaluated individuals:
	total --> 61
	real --> 61
___________________________________________


num_candidates is 75, to select 7
Problem calle_agustina_aroca_albacete_250_0_candidates_75_ns_7.json
\\\\\\\\\\\\\\\\\\\\\\\\